# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaikhabdullahwaseem17-byte/vigilant-meme/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Reading `skills/README.md` as instructed

In [ ]:
try:
    with open('/content/README (4).md', 'r') as f:
        readme_content = f.read()
    print(readme_content)
except FileNotFoundError:
    print('Error: /content/README (4).md not found. Please ensure the file exists in the correct path.')


# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's it.

## The table — find your task, load ONE skill

| Your task | Load this skill | Also load for data work |
|---|---|---|
| Any task — how to work with your assistant at all | `directing-your-ai-assistant/SKILL.md` | — |
| Pick a lane, frame your question (ML-02, ML-03) | `framing-ml-problems/SKILL.md` | `flyrank/flyrank-data/SKILL.md` |
| Write +

### Loading `writing-data-contracts/SKILL.md`

In [ ]:
try:
    with open('/content/SKILL.md', 'r') as f:
        skill_content = f.read()
    print(skill_content)
except FileNotFoundError:
    print('Error: /content/SKILL.md not found. Please ensure the file exists in the correct path.')


---
name: writing-data-contracts
description: Writes a data contract — what a row means, which fields are features vs labels vs context vs excluded, over which time windows — and verifies every claim with a query. Use before any feature building or modeling, or when results look wrong and the data definition is suspect.
---

# Writing data contracts

Most weak analysis fails here, before any model: nobody wrote down what a row means. A data
contract is that write-down — and a contract your code has checked, not just stated.

## The contract, section by section

1. **Unit of analysis.** One row = one WHAT? (a page? a page-day? a client?) State it.
2. **Time window.** Which dates does each field cover? Draw the windows if they differ.
3. **Field classification.** Every field you may touch goes in exactly one bucket:
   - **Feature** — knowable BEFORE the moment you predict, safe to use.
   - **Label / proxy** — the thing you predict, or what it is computed from. Never a feature.
   - **C

### Loading `flyrank/flyrank-data/SKILL.md`

In [ ]:
try:
    with open('/content/SKILL (1).md', 'r') as f:
        skill_content = f.read()
    print(skill_content)
except FileNotFoundError:
    print('Error: /content/SKILL (1).md not found. Please ensure the file exists in the correct path.')


---
name: flyrank-data
description: The FlyRank internship datasets — the 30k-row starter CSV and its gotchas, the ~79M-row warehouse release tables and grains, panel warnings, access, and iteration rules. Load for EVERY task that touches the data. (Project-specific: delete this folder when reusing the skill library elsewhere.)
---

# FlyRank internship data

Two datasets. The small one ships in this repo; the big one is hosted and gated.

## 1. Starter dataset (in this repo)

`data/raw/content_refresh_anonymized.csv` — 30,000 rows × 44 columns, one row per pseudonymized
content item, 32 clients, trailing-90-day metrics. Full column reference: `docs/data-dictionary.md`
(keep it open). The gotchas that cause 90% of mistakes:

- **Rate columns are ×100 percentages**: `ctr = 0.76` means 0.76%, not 76%. Applies to ctr,
  engagement_rate, scroll_rate, ai_traffic_pct, trend_pct.
- **`avg_position = 0` means "no data"**, not rank zero (1,205 rows).
- **`scroll_rate` and `ai_traffic_pct` can e

## 1. Unit of analysis + time window

One row = one what, over which dates? State it, then verify it below.

Unit of Analysis: One row represents the daily performance of a specific cntent item for a particular client.

Time Window spans from 2025-01-27 to 2026-06-30, covering approx 17 months.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



Features (knowable BEFORE prediction):
   gsc_avg_positio: Position metric (note 0 means no data, not rank zero).
   
     `ctr`: Click-through rate (×100 percentage).
  engagement_rate: User engagement (×100 percentage, can exceed 100).

   scroll_rate: How far users scroll (×100 percentage, can exceed 100).
   
   ai_traffic_pct: Percentage of traffic from AI sources (×100 percentage, can exceed 100).
   word_count: Content characteristic (from dim_content, if joined).

 content_type: Categorical content characteristic (from dim_content, if joined).

  has_keyword_data: A derived flag to handle missing keyword data based on content_type.

  ga4_data_available: Flag indicating GA4 data availability.

Label / Proxy (the thing to predict, or what it is computed from; never a feature):

   is_declining_label: The target variable to be predicted, explicitly derived from other metrics.

Context (for grouping, joining, splitting, reading; never for the model to learn from):

  report_date: For temporal grouping, joining, and splitting.
  
  client_id: Pseudonymized identifier for clients (IDs are never features).

  content_id: Pseudonymized identifier for content items (IDs are never features).

 client: Client name/identifier (often derived from client_id).
  
  content: Content identifier/name (often derived from content_id).

  gsc_data_start: Client-specific start date for GSC data (from dim_clients).

   ga4_data_start: Client-specific start date for GA4 data (from
   dim_clients).


Excluded (private, product-decision flags, or future information; each needs a one-line why):

trend_direction: Derived from trend_pct, which is used to compute is_declining_label. Including it would cause data leakage.

trend_pct: Directly used in the computation of is_declining_label. Including it would cause data leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Transitioning to Hugging Face Warehouse with DuckDB

As correctly pointed out, the ML-04 assignment utilizes the Hugging Face warehouse via DuckDB, not Google BigQuery. The previous BigQuery-specific cells were causing errors due to the incorrect setup.

Below, we will set up the necessary libraries and establish a connection to the Hugging Face dataset using DuckDB. You will need to add your Hugging Face token to Colab's secrets manager (the 🔑 icon on the left panel) and name it `HF_TOKEN`.

In [ ]:
# Install necessary libraries
!pip install duckdb huggingface_hub

In [ ]:
import duckdb
from huggingface_hub import HfFileSystem
from google.colab import userdata

# --- Set up Hugging Face Token and DuckDB Connection ---

# Load HF_TOKEN from Colab secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("Error: HF_TOKEN not found in Colab secrets. Please add your Hugging Face token.")
    HF_TOKEN = None

if HF_TOKEN:
    # Initialize Hugging Face FileSystem with the token
    fs = HfFileSystem(token=HF_TOKEN)

    # Connect to DuckDB
    con = duckdb.connect()

    # Install and load the httpfs extension for external file system access
    con.install_extension('httpfs')
    con.load_extension('httpfs')

    # Register the Hugging Face FileSystem with DuckDB
    con.register_filesystem(fs)

    # Define the Hugging Face base path for the dataset
    HF_DATASET_ROOT = "datasets/FlyRank/internship-warehouse"
    HF_TABLE_LOGICAL_PATH = f"{HF_DATASET_ROOT}/fact_content_daily_performance"

    # Use fs.glob to get all parquet file paths within the directory
    # fs.glob needs path without 'hf://' prefix
    all_parquet_files = fs.glob(f"{HF_TABLE_LOGICAL_PATH}/**/*.parquet")
    # Prepend 'hf://' prefix back for DuckDB's from_parquet to understand it's an fsspec path
    hf_full_parquet_paths = [f"hf://{p}" for p in all_parquet_files]

    if not hf_full_parquet_paths:
        print(f"Error: No parquet files found in hf://{HF_TABLE_LOGICAL_PATH}. Please check the path and permissions.")
        con = None # Indicate connection failed or dataset is empty
    else:
        # Create a DuckDB relation from these paths
        hf_relation = con.from_parquet(hf_full_parquet_paths)
        # Register this relation as a view for easier querying in subsequent cells
        hf_relation.create_view('fact_content_daily_performance_hf')

        print(f"Successfully connected to DuckDB with Hugging Face FileSystem and created view.")
        print(f"View 'fact_content_daily_performance_hf' created from {len(hf_full_parquet_paths)} parquet files.")
        print(f"Sample file: {hf_full_parquet_paths[0]}") # Show one file for confirmation
else:
    print("Hugging Face token not available. Cannot establish DuckDB connection to Hugging Face dataset.")
    con = None # Indicate connection failed

Successfully connected to DuckDB with Hugging Face FileSystem and created view.
View 'fact_content_daily_performance_hf' created from 18 parquet files.
Sample file: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet


### 3.1 Verify Grain with DuckDB: `report_date × client × content` is unique

In [ ]:
# Verify the grain of the `fact_content_daily_performance` table using DuckDB.
# This query checks if any combination of report_date, client, and content appears more than once.
# As per the skill, an empty result means the grain holds.

if con is not None:
    grain_query_duckdb = f"""SELECT
        report_date,
        client_hash_id,
        content_hash_id, -- Corrected column name
        COUNT(*) AS num_duplicates
    FROM
        fact_content_daily_performance_hf
    GROUP BY
        report_date, client_hash_id, content_hash_id -- Corrected column name
    HAVING
        num_duplicates > 1
    LIMIT 5
    """

    duplicate_rows_df_duckdb = con.query(grain_query_duckdb).fetchdf()

    if duplicate_rows_df_duckdb.empty:
        print("Grain verification successful: No duplicate rows found for (report_date, client_hash_id, content_hash_id).")
    else:
        print("Grain verification failed: Duplicates found!")
        display(duplicate_rows_df_duckdb)
else:
    print("DuckDB connection not established or dataset not loaded. Please ensure HF_TOKEN is set and the connection cell ran successfully.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain verification failed: Duplicates found!


,report_date,client_hash_id,content_hash_id,num_duplicates
0,2026-06-14,client_1a8bf67cad4ee525,content_3084acbc2aca184b,2
1,2026-06-21,client_810019792c9b8efc,content_41a28b3332556857,2
2,2026-06-22,client_810019792c9b8efc,content_a44a3f299cba632b,2
3,2026-06-29,client_def0955f7a377868,content_fa5e8c5d22b24cf0,2
4,2026-06-13,client_8ddc46da5414ffd8,content_1d06a2c99a935e49,2


#### Grain Verification Result: Duplicates Found!

The initial grain verification query `report_date × client × content` has found duplicate entries. This means the assumption that this combination uniquely identifies a row is incorrect for at least some rows in the dataset. This finding is crucial and will need to be addressed in the data contract, potentially by redefining the unit of analysis or by understanding the source of these duplicates.


### 3.2 Verify Counts and Time Window with DuckDB

In [ ]:
# Self-healing cell: Ensures DuckDB connection and 'fact_content_daily_performance_hf' view are active.
# This prevents 'CatalogException' if the kernel restarts or cells are run out of order.

import duckdb
from huggingface_hub import HfFileSystem
from google.colab import userdata

# Check if 'con' object exists in global scope and if it's a DuckDBPyConnection
# Also check if the view 'fact_content_daily_performance_hf' is registered
view_exists = False
if 'con' in globals() and isinstance(con, duckdb.DuckDBPyConnection):
    try:
        # Check if the view exists by querying the duckdb_views catalog
        view_check_query = "SELECT count(*) FROM duckdb_views WHERE view_name = 'fact_content_daily_performance_hf'"
        if con.query(view_check_query).fetchone()[0] > 0:
            view_exists = True
    except Exception as e:
        print(f"Error checking for view existence: {e}. Assuming view does not exist.")

if not view_exists or 'con' not in globals() or not isinstance(con, duckdb.DuckDBPyConnection):
    print("DuckDB connection or 'fact_content_daily_performance_hf' view not found/active. Re-establishing connection and view...")
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except userdata.SecretNotFoundError:
        print("Error: HF_TOKEN not found in Colab secrets. Please add your Hugging Face token.")
        HF_TOKEN = None

    if HF_TOKEN:
        fs = HfFileSystem(token=HF_TOKEN)
        con = duckdb.connect()
        con.install_extension('httpfs')
        con.load_extension('httpfs')
        con.register_filesystem(fs)

        HF_DATASET_ROOT = "datasets/FlyRank/internship-warehouse"
        HF_TABLE_LOGICAL_PATH = f"{HF_DATASET_ROOT}/fact_content_daily_performance"

        all_parquet_files = fs.glob(f"{HF_TABLE_LOGICAL_PATH}/**/*.parquet")
        hf_full_parquet_paths = [f"hf://{p}" for p in all_parquet_files]

        if not hf_full_parquet_paths:
            print(f"Error: No parquet files found in hf://{HF_TABLE_LOGICAL_PATH}.")
            con = None
        else:
            hf_relation = con.from_parquet(hf_full_parquet_paths)
            hf_relation.create_view('fact_content_daily_performance_hf')
            print("DuckDB connection and view re-established successfully.")
    else:
        print("Hugging Face token not available. Cannot re-establish DuckDB connection.")
        con = None
else:
    print("DuckDB connection and 'fact_content_daily_performance_hf' view already active.")


DuckDB connection and 'fact_content_daily_performance_hf' view already active.


In [ ]:
import pandas as pd

# Verify total row count and the time window (min/max report_date) using DuckDB.
# Expected total rows: ~78,835,655
# Expected date range: 2025-01-27 to 2026-06-30

if con is not None:
    counts_query_duckdb = f"""SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM
        fact_content_daily_performance_hf
    """

    counts_df_duckdb = con.query(counts_query_duckdb).fetchdf()
    display(counts_df_duckdb)

    # Optional: Add checks against expected values
    expected_total_rows = 78835655
    expected_min_date = pd.to_datetime('2025-01-27').date()
    expected_max_date = pd.to_datetime('2026-06-30').date()

    if not counts_df_duckdb.empty:
        actual_total_rows = counts_df_duckdb['total_rows'].iloc[0]
        actual_min_date = pd.to_datetime(counts_df_duckdb['min_report_date'].iloc[0]).date() # Convert to date for comparison
        actual_max_date = pd.to_datetime(counts_df_duckdb['max_report_date'].iloc[0]).date() # Convert to date for comparison

        print(f"\nExpected Total Rows: {expected_total_rows}, Actual: {actual_total_rows}")
        print(f"Expected Min Date: {expected_min_date}, Actual: {actual_min_date}")
        print(f"Expected Max Date: {expected_max_date}, Actual: {actual_max_date}")

        if actual_total_rows == expected_total_rows:
            print("Row count matches expectations.")
        else:
            print("WARNING: Row count does NOT match expectations.")

        if actual_min_date == expected_min_date and actual_max_date == expected_max_date:
            print("Time window matches expectations.")
        else:
            print("WARNING: Time window does NOT match expectations.")
else:
    print("DuckDB connection not established or dataset not loaded. Please ensure HF_TOKEN is set and the connection cell ran successfully.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,min_report_date,max_report_date
0,78835655,2025-01-27,2026-06-30



Expected Total Rows: 78835655, Actual: 78835655
Expected Min Date: 2025-01-27, Actual: 2025-01-27
Expected Max Date: 2026-06-30, Actual: 2026-06-30
Row count matches expectations.
Time window matches expectations.


### 3.3 Initial Missing Values Check for Key Columns with DuckDB

In [ ]:
# Calculate the percentage of missing values for key columns using DuckDB.
# This helps identify columns with significant missingness that might require imputation or special handling.

if con is not None:
    missing_values_query_duckdb = f"""SELECT
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS gsc_avg_position_missing_pct,
        SUM(CASE WHEN scroll_events IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS scroll_events_missing_pct
    FROM
        fact_content_daily_performance_hf
    """

    missing_values_df_duckdb = con.query(missing_values_query_duckdb).fetchdf()
    display(missing_values_df_duckdb)

    print("\nNote: The 'gsc_avg_position = 0' is a specific 'no data' case, not NULL. This query only checks for actual NULLs.")
    print("Further investigation would be needed for patterns of missingness, e.g., grouped by content_type.")
    print("Columns like 'ctr', 'engagement_rate', 'ai_traffic_pct', 'is_declining_label' were not found in this table for a direct missingness check. They might be in other tables or require derivation.")
else:
    print("DuckDB connection not established or dataset not loaded. Please ensure HF_TOKEN is set and the connection cell ran successfully.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_avg_position_missing_pct,scroll_events_missing_pct
0,63.252667,37.591274



Note: The 'gsc_avg_position = 0' is a specific 'no data' case, not NULL. This query only checks for actual NULLs.
Further investigation would be needed for patterns of missingness, e.g., grouped by content_type.
Columns like 'ctr', 'engagement_rate', 'ai_traffic_pct', 'is_declining_label' were not found in this table for a direct missingness check. They might be in other tables or require derivation.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

  Unbalanced History: History depth differs significantly per client. Relying on a global calendar window for analysis will be misleading; per-client windows should be preferred. Rows before a clients gsc_data_start or ga4_data_start simply do not exist or are filled with zeros, which is not equivalent to 'no engagement' in a meaningful historical sense.


GA4 Data Zero-Filling: Rows before a client's ga4_data_start have GA4 columns zero-FILLED with ga4_data_available = FALSE. This means '0' for GA4 metrics does not always mean zero engagement, but rather that data was not collected, preventing insights into actual engagement during those periods.


Limited Search/Analytics History: A significant portion of clients have little to no usable search/analytics history. This limits the generalizability of findings across all clients and requires careful filtering.


  GSC Average Position 'No Data': gsc_avg-position = 0 signifies 'no data' or 'not ranked', not an actual rank of zero (which is impossible). This requires careful handling to avoid misinterpreting zero values as strong performance.


  Lagged Data: The data represents past observations. It inherently cannot predict future events or new trends that are not reflected in historical patterns. For example, it cannot tell us the impact of a completely new content type or a sudden market shift not covered in the historical window

## 5. Output

*What the analysis hands to the human, in one sentence.*

The analysis will provide a databacked understanding of content performance trends for each client, identifying content items that are experiencing decline and offering insights into potential contributing factors based on the defined features and context, enabling informed editorial and optimization decisions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.